In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv("../data/context_merged_dataset.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (10000, 20)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,timestamp,Ambient_Temperature,Load_Density,Humidity,Shift,Day_Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,2025-01-01 00:00:00,26,62,41,Evening,Weekday
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,2025-01-01 00:01:00,39,34,47,Morning,Weekend
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,2025-01-01 00:02:00,34,43,81,Evening,Weekday
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,2025-01-01 00:03:00,30,97,46,Evening,Weekend
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,2025-01-01 00:04:00,27,76,49,Morning,Weekday


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
 14  timestamp                10000 non

In [4]:
df.isnull().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
timestamp                  0
Ambient_Temperature        0
Load_Density               0
Humidity                   0
Shift                      0
Day_Type                   0
dtype: int64

In [5]:
df.columns = [
    c.replace(" ", "_")
     .replace("[", "")
     .replace("]", "")
     .replace("/", "_")
    for c in df.columns
]

print(df.columns.tolist())

['UDI', 'Product_ID', 'Type', 'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Machine_failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift', 'Day_Type']


In [6]:
features = [
    "Type",
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",
    "Ambient_Temperature",
    "Load_Density",
    "Humidity",
    "Shift",
    "Day_Type"
]

X = df[features].copy()

y = df["Machine_failure"]

In [7]:
print(df["Type"].unique())

print(df["Shift"].unique())

print(df["Day_Type"].unique())

<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str
<StringArray>
['Evening', 'Morning', 'Night']
Length: 3, dtype: str
<StringArray>
['Weekday', 'Weekend']
Length: 2, dtype: str


In [8]:
encoders = {}

categorical_columns = [
    "Type",
    "Shift",
    "Day_Type"
]

for col in categorical_columns:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(X[col])

    encoders[col] = encoder

print(X.head())

   Type  Air_temperature_K  Process_temperature_K  Rotational_speed_rpm  \
0     2              298.1                  308.6                  1551   
1     1              298.2                  308.7                  1408   
2     1              298.1                  308.5                  1498   
3     1              298.2                  308.6                  1433   
4     1              298.2                  308.7                  1408   

   Torque_Nm  Tool_wear_min  Ambient_Temperature  Load_Density  Humidity  \
0       42.8              0                   26            62        41   
1       46.3              3                   39            34        47   
2       49.4              5                   34            43        81   
3       39.5              7                   30            97        46   
4       40.0              9                   27            76        49   

   Shift  Day_Type  
0      0         0  
1      1         1  
2      0         0  
3      0

In [9]:
joblib.dump(encoders, "../models/encoders.pkl")

print("Encoders Saved Successfully")

Encoders Saved Successfully


In [10]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

print("Training Shape :", X_train.shape)

print("Testing Shape :", X_test.shape)

Training Shape : (8000, 11)
Testing Shape : (2000, 11)


In [11]:
model = LGBMClassifier(

    random_state=42,

    n_estimators=100,

    learning_rate=0.1

)

model.fit(X_train, y_train)

print("Model trained successfully.")

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001518 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1077
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.033875 -> initscore=-3.350616
[LightGBM] [Info] Start training from score -3.350616
Model trained successfully.


In [12]:
pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))

print("Precision :", precision_score(y_test, pred))

print("Recall :", recall_score(y_test, pred))

print("F1 Score :", f1_score(y_test, pred))

Accuracy : 0.987
Precision : 0.92
Recall : 0.6764705882352942
F1 Score : 0.7796610169491526


In [13]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.92      0.68      0.78        68

    accuracy                           0.99      2000
   macro avg       0.95      0.84      0.89      2000
weighted avg       0.99      0.99      0.99      2000



In [14]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                  Feature  Importance
4               Torque_Nm         547
5           Tool_wear_min         479
3    Rotational_speed_rpm         406
1       Air_temperature_K         384
2   Process_temperature_K         342
7            Load_Density         279
8                Humidity         224
6     Ambient_Temperature         184
0                    Type          77
9                   Shift          51
10               Day_Type          27


In [15]:
joblib.dump(model, "../models/model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [16]:
print(model.feature_name_)

print()

print("Total Features :", len(model.feature_name_))

['Type', 'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift', 'Day_Type']

Total Features : 11


In [17]:
sample = X.iloc[[0]]

prediction = model.predict(sample)

probability = model.predict_proba(sample)

print("Prediction :", prediction[0])

print("Failure Probability :", probability[0][1])

Prediction : 0
Failure Probability : 6.735189923983327e-05


In [19]:
import joblib

encoders = joblib.load("../models/encoders.pkl")

print(encoders["Shift"].classes_)
print(encoders["Day_Type"].classes_)
print(encoders["Type"].classes_)

['Evening' 'Morning' 'Night']
['Weekday' 'Weekend']
['H' 'L' 'M']
